# German Credit Analysis (scorecardpl)

This notebook demonstrates end-to-end usage of `scorecardpl` on the classic German Credit dataset:
- Load and prepare the data
- Train/valid split + variable filtering
- WOE/IV binning and inspection
- Logistic-regression scorecard + AUC/KS, scores
- Optional: SHAP-based scorecard using a RandomForest


Note: SHAP section is compatible with SHAP 0.48+ and handles the new values shape with a robust fallback.


In [1]:
!pip install ../

Processing /home/zaenal/personal_project/scorecardpl
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scorecardpl: filename=scorecardpl-0.1.1-py3-none-any.whl size=28025 sha256=4f7f84fdb68182ca7154d26f9ecd5ee654725b4bba842369780296ae34390cac
  Stored in directory: /tmp/pip-ephem-wheel-cache-u8pzn8mq/wheels/e4/e6/ea/f1b986afcd60f8c0a0298c3738f6bd36ef57166a6ab6df3ea4
Successfully built scorecardpl
  Attempting uninstall: scorecardpl
    Found existing installation: scorecardpl 0.1.1
    Uninstalling scorecardpl-0.1.1:
      Successfully uninstalled scorecardpl-0.1.1


In [2]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import polars as pl

from scorecardpl import (
    split_df, var_filter, woebin, woebin_ply,
    scorecard, scorecard_ply, perf_eva, iv_summary,
    woebin_plot, bins_export_json, bins_import_json,
)

# Optional: SHAP-based scorecard support
HAS_SHAP = False
try:
    import shap  # noqa: F401
    from sklearn.ensemble import RandomForestClassifier
    from scorecardpl import scorecard_shap, SHAPScorecardModel
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

# Ensure a local plots dir for saved figures
os.makedirs('plots', exist_ok=True)


## Load German Credit data

We fetch the Statlog German Credit dataset from UCI. The label maps to 0/1 as: 1=good (0), 2=bad (1).

In [3]:
uci_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
cols = [
    'Status', 'Duration', 'CreditHistory', 'Purpose', 'CreditAmount',
    'Savings', 'Employment', 'InstallmentRate', 'PersonalStatusSex', 'OtherDebtors',
    'ResidenceSince', 'Property', 'Age', 'OtherInstallmentPlans', 'Housing',
    'ExistingCredits', 'Job', 'Liables', 'Telephone', 'ForeignWorker', 'class'
]
df_pd = pd.read_csv(uci_url, sep=' ', header=None, names=cols)
# Map target: 1 (good) -> 0, 2 (bad) -> 1
df_pd['y'] = (df_pd['class'] == 2).astype(int)
df_pd = df_pd.drop(columns=['class'])
df_pl = pl.from_pandas(df_pd)
df_pl.head()


Status,Duration,CreditHistory,Purpose,CreditAmount,Savings,Employment,InstallmentRate,PersonalStatusSex,OtherDebtors,ResidenceSince,Property,Age,OtherInstallmentPlans,Housing,ExistingCredits,Job,Liables,Telephone,ForeignWorker,y
str,i64,str,str,i64,str,str,i64,str,str,i64,str,i64,str,str,i64,str,i64,str,str,i64
"""A11""",6,"""A34""","""A43""",1169,"""A65""","""A75""",4,"""A93""","""A101""",4,"""A121""",67,"""A143""","""A152""",2,"""A173""",1,"""A192""","""A201""",0
"""A12""",48,"""A32""","""A43""",5951,"""A61""","""A73""",2,"""A92""","""A101""",2,"""A121""",22,"""A143""","""A152""",1,"""A173""",1,"""A191""","""A201""",1
"""A14""",12,"""A34""","""A46""",2096,"""A61""","""A74""",2,"""A93""","""A101""",3,"""A121""",49,"""A143""","""A152""",1,"""A172""",2,"""A191""","""A201""",0
"""A11""",42,"""A32""","""A42""",7882,"""A61""","""A74""",2,"""A93""","""A103""",4,"""A122""",45,"""A143""","""A153""",1,"""A173""",2,"""A191""","""A201""",0
"""A11""",24,"""A33""","""A40""",4870,"""A61""","""A73""",3,"""A93""","""A101""",4,"""A124""",53,"""A143""","""A153""",2,"""A173""",2,"""A191""","""A201""",1


## Train/valid split and variable filtering

In [4]:
y = 'y'
xs = [c for c in df_pl.columns if c != y]
train, valid = split_df(df_pl, y=y, test_size=0.3, random_state=42)
train = var_filter(train, y=y, x=xs)
train.shape, valid.shape


((700, 21), (300, 21))

## WOE/IV binning

In [5]:
bins = woebin(
    train, y=y, x=[c for c in train.columns if c != y],
    bins=6, method='chi2', chi2_params={'init_bins': 60},
    monotonic='auto', cat_max_bins=5,
)
ivsum = iv_summary(bins)
ivsum


variable,iv,nbin
str,f64,i64
"""Status""",0.809315,4
"""CreditAmount""",0.58496,3
"""CreditHistory""",0.326533,5
"""Savings""",0.232083,5
"""Duration""",0.221583,3
…,…,…
"""ResidenceSince""",0.013592,3
"""Telephone""",0.012284,2
"""ExistingCredits""",0.00422,2


### WOE plots (optional)

In [6]:
# Save a couple of WOE plots to the local 'plots' folder
woebin_plot(bins, var='CreditAmount', save_dir='plots', show=False)
woebin_plot(bins, var='Status', save_dir='plots', show=False)
sorted([p for p in os.listdir('plots') if p.startswith('woe_')])


['woe_CreditAmount.png', 'woe_Status.png']

## Logistic-regression scorecard

In [7]:
train_w = woebin_ply(train, bins)
valid_w = woebin_ply(valid, bins)
sc = scorecard(bins, y=y, data=train_w)
# Predict proba on validation from WOE features
X_valid = valid_w.select([c for c in valid_w.columns if c.endswith('_woe')]).to_numpy()
proba = sc.model.predict_proba(X_valid)[:, 1]
perf_lr = perf_eva(valid_w[y], proba, plot='both', save_prefix='plots/german_credit')
perf_lr


{'auc': 0.7500529100529102, 'ks': 0.43968253968253973}

### Score application

In [8]:
scores_lr = scorecard_ply(valid_w, sc.points_map)
scores_lr.head(10)


score
f64
806.671901
783.740701
823.076376
745.967776
793.40305
795.827715
824.888579
788.214948
765.297321


In [25]:
scores_lr.min()

644.9604568522393

In [26]:
scores_lr.max()

1290.9266199540468

## Optional: SHAP-based scorecard (RandomForest)

In [13]:
if HAS_SHAP:
    est = RandomForestClassifier(n_estimators=200, max_depth=3, random_state=42)
    sc_shap = scorecard_shap(
        bins=bins, y=y, data=train, estimator=est,
        shap_sample_n=2000, pdo=50.0, base_score=650.0, odds=20.0,
    )
    proba_rf = sc_shap.predict_proba(valid, bins)
    perf_rf = perf_eva(valid[y], proba_rf, plot='both', save_prefix='plots/german_credit_rf_shap')
    perf_rf
else:
    print('SHAP not available. To enable: pip install shap')


### SHAP-based points

In [14]:
if HAS_SHAP:
    scores_rf = sc_shap.predict_points(valid, bins)
    scores_rf.head(10)
else:
    print('Skip: SHAP not installed')


In [24]:
scores_rf.max()

860.9639614898426

In [16]:
sc_shap

SHAPScorecardModel(estimator=RandomForestClassifier(max_depth=3, n_estimators=200, random_state=42), points_map={'Status': shape: (4, 4)
┌──────────┬─────┬─────┬───────────┐
│ variable ┆ bin ┆ woe ┆ points    │
│ ---      ┆ --- ┆ --- ┆ ---       │
│ str      ┆ str ┆ f64 ┆ f64       │
╞══════════╪═════╪═════╪═══════════╡
│ Status   ┆ A11 ┆ 0.0 ┆ -5.563085 │
│ Status   ┆ A12 ┆ 0.0 ┆ -3.944303 │
│ Status   ┆ A13 ┆ 0.0 ┆ 3.819501  │
│ Status   ┆ A14 ┆ 0.0 ┆ 6.026827  │
└──────────┴─────┴─────┴───────────┘, 'Duration': shape: (3, 4)
┌──────────┬─────────────┬─────┬───────────┐
│ variable ┆ bin         ┆ woe ┆ points    │
│ ---      ┆ ---         ┆ --- ┆ ---       │
│ str      ┆ str         ┆ f64 ┆ f64       │
╞══════════╪═════════════╪═════╪═══════════╡
│ Duration ┆ (-inf, 9.0] ┆ 0.0 ┆ 3.034048  │
│ Duration ┆ (11.0, inf] ┆ 0.0 ┆ -0.700473 │
│ Duration ┆ (9.0, 11.0] ┆ 0.0 ┆ 3.456044  │
└──────────┴─────────────┴─────┴───────────┘, 'CreditHistory': shape: (5, 4)
┌───────────────┬─────┬─────┬